In [ ]:
from dataclasses import dataclass
from src import *
from src.internals import *
from src.shading import BlinnPhongShader
from dataclasses import field

# Integrators
---
### Simple ray caster

The `RayCaster` traces a ray into the scene and computes the color only at the first visible intersection. It does not generate any secondary rays, so it performs only local shading without recursion.

This makes it the simplest integrator in the system and a good starting point for understanding how ray tracing works. If a ray hits an object, the color is computed using the selected shading model and all scene lights. If no intersection is found, the background color is returned instead.

Because no recursive rays are traced, all lighting can be evaluated directly at the first hit point without needing to handle reflections or refractions.

$$
C = C_{local}
$$

Where:
- $C$ is the final color returned by the integrator,
- $C_{local}$ is the color computed by local shading at the first hit point.

In [ ]:
@dataclass
class MyRayCaster(Integrator):
    """
    A simple ray caster that only computes local shading at the first hit point, without any recursion.
    """
    scene: Scene
    shader: LocalShading | None = None

    def __post_init__(self):
        if self.shader is None:
            self.shader = BlinnPhongShader()

    def cast_ray(self, ray: Ray, depth: int | None = None) -> Color:
        hit = self.scene.intersect(ray)

        if hit is not None:
            return self.shader.shade_multiple_lights(
                hit,
                lights=self.scene.lights,
                view_dir=-ray.direction,   # ray to camera is opposite of ray direction
                scene=self.scene
            )

        return Color.background_color(ray.direction, self.scene.skybox) # if ray misses, return background color

#### lets set up a scene to test this out

In [ ]:
point_light = PointLight(position=Vertex(5, 5, 0), intensity=1)
ambient_light = AmbientLight(intensity=0.2)
lights = [point_light, ambient_light]

objects = [
    Object(
        geometry=Sphere(),
        material=PhongMaterial(base_color=Color(1, 0, 0), reflectivity=0.7),
    ).translate(0, 0.5, -5),
    Object(
        geometry=Plane(),
        material=PhongMaterial(base_color=Color(0.5, 0.5, 0.5), reflectivity=0.2),
    ).translate(0, -1, 0),
]

camera=PinholeCamera(
    origin=Vertex(0, 0, 0),
    direction=Vector(0, 0, -1),
    fov_deg=50,
)

scene = Scene(
    objects=objects,
    lights=lights,
    camera=camera,
)

ray_caster = MyRayCaster(scene=scene)
rt = LinearRenderLoop(
    scene=scene,
    integrator=ray_caster,
    render_config=RenderConfig(
        resolution=Resolution.R360p,
    ),
    preview_config=PreviewConfig(
        progress_display=ProgressDisplay.TQDM_IMAGE_PREVIEW,
    )
)

#### Rendering with the ray caster integrator

In [ ]:
rt.render("images/ray_caster_demo.png")
ipynb_display_images("images/ray_caster_demo.png")

### Reflection integrator

The `ReflectionIntegrator` adds recursive reflections to the ray caster. After a ray hits a surface, it first computes the local color. If the material is reflective, it then generates a reflected ray and traces it recursively.

The final color is computed as a blend of the local shading and the reflected contribution based on the material reflectance. The `max_depth` parameter limits the number of ray bounces, while `bias` slightly offsets the origin of the reflected ray to avoid self-intersections.

$$
C = (1 - R) \cdot C_{local} + R \cdot C_{reflected}
$$

Where:
- $C$ is the final color returned by the integrator,
- $R$ is the material reflectance at the hit point,
- $C_{local}$ is the color from local shading,
- $C_{reflected}$ is the color obtained by recursively tracing the reflected ray.

The reflected ray direction is computed as:

$$
\mathbf{r} = \mathbf{d} - 2(\mathbf{d} \cdot \mathbf{n})\mathbf{n}
$$

Where:
- $\mathbf{d}$ is the incoming ray direction,
- $\mathbf{n}$ is the surface normal at the hit point.

> **note**
> - In this implementation, the variable `depth` represents the remaining recursion budget rather than the current bounce index. The primary camera ray therefore starts with `depth = max_depth`, and each reflected or refracted ray decreases this value by one for remaining depth.

In [ ]:
@dataclass
class MyReflectionIntegrator(Integrator):
    """
    A simple reflection integrator that computes local shading at the first hit point and recursively traces reflected rays based on material reflectance.
    """
    scene: Scene
    shader: LocalShading = field(default_factory=BlinnPhongShader)
    max_depth: int = 3
    bias: float = 1e-3

    def cast_ray(self, ray: Ray, depth: int | None = None) -> Color:
        if depth is None:
            depth = self.max_depth

        hit = self.scene.intersect(ray)
        if hit is None:
            return Color.background_color(ray.direction, skybox=self.scene.skybox)

        local_color = self.shader.shade_multiple_lights(
            hit,
            lights=self.scene.get_all_lights(),
            view_dir=-ray.direction,
            scene=self.scene,
        )

        # return local color if we have reached the maximum recursion depth or if the material is not reflective
        reflectivity = hit.material.get_reflectance()
        if depth <= 0 or reflectivity <= 0.0:
            return local_color

        # compute the reflected ray direction using the surface normal and incoming ray direction
        normal = hit.geom.normal.normalize()
        n = normal if normal.dot(ray.direction) < 0.0 else -normal

        # compute the reflected ray its color contribution by recursively tracing it through the scene
        reflected_dir = reflect_dir(ray.direction, n).normalize()
        reflected_ray = Ray(hit.geom.point + n * self.bias, reflected_dir)
        reflected_color = self.cast_ray(reflected_ray, depth - 1)

        # blend the local color and reflected color based on the material reflectance
        return local_color * (1.0 - reflectivity) + reflected_color * reflectivity

def reflect_dir(v: Vector, n: Vector) -> Vector:
    """
    Reflects the incoming vector `v` around the normal `n` using the formula: r = v - 2 * (v . n) * n
    :param v: the incoming vector
    :param n: the normal vector at the hit point, should be normalized and facing against the incoming vector
    :return: the reflected vector direction
    """
    return v - n * (2.0 * v.dot(n))

### Reflective objects in the scene depth 0 - local shading only
- At recursion depth 0, only the local shading at the first visible hit point is evaluated. Reflection rays are not traced yet, so reflective effects are not visible. The result is therefore equivalent to the simple ray caster shown above.

In [ ]:
integrator_reflect = MyReflectionIntegrator(scene=scene)
rt_reflect = LinearRenderLoop(
    scene=scene,
    integrator=integrator_reflect,
    render_config=RenderConfig(
        resolution=Resolution.R360p,
        max_depth=0,
    ),
    preview_config=PreviewConfig(
        progress_display=ProgressDisplay.TQDM_IMAGE_PREVIEW,
    )
)
img = rt_reflect.render("images/reflection_integrator_demo.png")
ipynb_display_images(img)

### Reflective objects in the scene depth 5 - recursive reflections visible

At recursion depth 5, reflection rays are traced recursively through the scene. This allows reflective objects to show colors and shapes of other visible surfaces, including repeated reflections between reflective objects. The result is no longer limited to local shading, because each reflected ray can contribute additional light from the surfaces it reaches.

> **Try it yourself** create scene with multiple reflective objects and experiment with different `max_depth` values to see how it affects the visibility of reflections. You can also adjust the reflectivity of materials to see how it influences the blending between local shading and reflected contributions.

In [ ]:
rt_reflect = LinearRenderLoop(
    scene=scene,
    integrator=integrator_reflect,
    render_config=RenderConfig(
        resolution=Resolution.R360p,
        max_depth=5,
    ),
    preview_config=PreviewConfig(
        progress_display=ProgressDisplay.TQDM_IMAGE_PREVIEW,
    )
)
img = rt_reflect.render("images/reflection_integrator_demo.png")
ipynb_display_images(img)

### Reflection and refraction integrator based on Whitted's ray tracing algorithm

This integrator implements a simplified Whitted-style recursive ray tracing algorithm. It first finds the closest intersection of the primary ray and evaluates local illumination at that point using the selected shader. If the material is neither reflective nor transparent, this local contribution is returned directly.

In this implementation, opaque reflective materials and transparent materials are handled as two separate cases. For opaque reflective surfaces, the explicit material reflectivity is used to blend the local surface color with the recursively traced reflected color:

$$
C = (1 - r) C_{\mathrm{local}} + r C_{\mathrm{reflected}},
$$

where $r$ is the material reflectivity.

For transparent materials, the explicit reflectivity parameter is not used directly. Instead, reflection and refraction are split using the Fresnel coefficient. The local contribution is first reduced according to the material transparency, and the transparent part is then divided between the reflected and refracted rays:

$$
C = (1 - t) C_{\mathrm{local}}
  + t \left( k_r C_{\mathrm{reflected}} + (1 - k_r) C_{\mathrm{refracted}} \right),
$$

where:

- $C$ is the final color returned by the integrator,
- $C_{\mathrm{local}}$ is the color computed by local shading,
- $C_{\mathrm{reflected}}$ is the color obtained by recursively tracing the reflected ray,
- $C_{\mathrm{refracted}}$ is the color obtained by recursively tracing the refracted ray,
- $t$ is the material transparency,
- $k_r$ is the Fresnel reflectance coefficient.

If refraction is not possible because of total internal reflection, the refracted contribution is omitted and the transparent contribution is assigned fully to reflection:

$$
C = (1 - t) C_{\mathrm{local}} + t C_{\mathrm{reflected}}.
$$

This separation is a didactic simplification. A more physically based model would usually treat reflection as a Fresnel-dependent property of the material interface and would combine reflection, refraction, and absorption in a more consistent energy model.

A small bias is applied to the origin of secondary rays to reduce self-intersection artifacts. Reflection rays are shifted slightly away from the surface, while refraction rays are shifted to the opposite side of the surface.

The Fresnel reflectance coefficient is approximated using Schlick's approximation:

$$
k_r(\theta) = R_0 + (1 - R_0)(1 - \cos \theta)^5,
$$

where

$$
R_0 = \left( \frac{n_1 - n_2}{n_1 + n_2} \right)^2.
$$

Here, $n_1$ and $n_2$ are the indices of refraction of the two media. In this implementation, the interface is simplified to air and the material. The value $\theta$ represents the angle between the surface normal and the view direction, where the view direction is opposite to the incoming ray direction.

Schlick's approximation is used because it is simple and computationally inexpensive, while still capturing the main Fresnel effect: reflection becomes stronger at grazing angles.

> **Note:** Shadow testing is implemented inside the shader for educational purposes. The integrator therefore does not need to handle shadow rays directly. When computing the contribution of each light source, the shader checks whether the light is occluded. This keeps the integrator focused on recursive reflection and refraction instead of separately managing shadow rays and multiple light sources.

In [ ]:
@dataclass
class MyWhittedIntegrator(Integrator):
    """
    A simple Whitted-style integrator that computes local shading at the first hit point and recursively traces reflected and refracted rays based on material properties.
    """
    scene: Scene
    shader: LocalShading = field(default_factory=BlinnPhongShader) # default shader to use for local shading at each hit point
    max_depth: int = 3
    bias: float = 1e-3

    def cast_ray(self, ray: Ray, depth: int | None = None) -> Color:
        if depth is None:
            depth = self.max_depth

        hit = self.scene.intersect(ray)
        if hit is None:
            return Color.background_color(ray.direction, skybox=self.scene.skybox)

        local_color = self.shader.shade_multiple_lights(
            hit, lights=self.scene.get_all_lights(), view_dir=-ray.direction, scene=self.scene
        )

        if depth <= 0:
            return local_color

        material = hit.material
        reflectivity = material.get_reflectance()
        transparency = material.get_transparency()

        result = local_color
        if reflectivity <= 0.0 and transparency <= 0.0:
            return result

        # here would be normal modification if we want to support normal perturbation for more complex materials, but for simplicity we will just use the geometry normal
        normal = hit.geom.normal.normalize()
        point = hit.geom.point

        if transparency > 0.0:
            result = local_color * (1.0 - transparency)

            reflected_ray = self.make_reflection_ray(ray, point, normal)
            reflected_color = self.cast_ray(reflected_ray, depth - 1)

            kr = self._fresnel(ray, normal, material)
            refracted_ray = self.make_refraction_ray(ray, point, normal, material)

            # if refraction is not possible (total internal reflection)
            if refracted_ray is None:
                result += reflected_color * transparency
            # if refraction is possible, we need to trace the refracted ray and blend its contribution with the reflected ray based on the Fresnel coefficient
            else:
                refracted_color = self.cast_ray(refracted_ray, depth - 1)
                result += reflected_color * (transparency * kr)
                result += refracted_color * (transparency * (1.0 - kr))

            return result

        # if the material is not transparent but is reflective, we only need to trace the reflected ray and blend it with the local color based on the reflectivity
        if reflectivity > 0.0:
            reflected_ray = self.make_reflection_ray(ray, point, normal)
            reflected_color = self.cast_ray(reflected_ray, depth - 1)
            return local_color * (1.0 - reflectivity) + reflected_color * reflectivity

        return result

    @staticmethod
    def _fresnel(ray: Ray, normal: Vector, material: Material) -> float:
        # Compute the Fresnel reflectance coefficient using Schlick's approximation
        ior_m = material.get_ior()
        front_face = normal.dot(ray.direction) < 0.0
        n = normal if front_face else -normal
        # For didactic simplicity, we assume an interface between air and the material.
        ior_out, ior_in = (1.0, ior_m) if front_face else (ior_m, 1.0)
        return fresnel_schlick(ray.direction, n, ior_out=ior_out, ior_in=ior_in)

    def make_reflection_ray(self, ray: Ray, point: Vertex, normal: Vector) -> Ray:
        # Flip normal to ensure it is facing against the ray direction if necessary
        n = normal if normal.dot(ray.direction) < 0.0 else -normal
        direction = reflect_dir(ray.direction, n).normalize()
        return Ray(point + n * self.bias, direction)

    def make_refraction_ray(self, ray: Ray, point: Vertex, normal: Vector, material) -> Ray | None:
        # Flip normal to ensure it is facing against the ray direction if necessary
        ior_m = material.get_ior()
        front_face = normal.dot(ray.direction) < 0.0
        n = normal if front_face else -normal
        # For didactic simplicity, we assume an interface between air and the material.
        ior_out, ior_in = (1.0, ior_m) if front_face else (ior_m, 1.0)
        # use the refract function to compute the refracted ray direction. Based on snell's law, if the ray cannot be refracted (total internal reflection), the function will return None.
        direction = refract(ray.direction, n, ior_out=ior_out, ior_in=ior_in)
        if direction is None:
            return None
        return Ray(point - n * self.bias, direction.normalize())

def reflect_dir(v: Vector, n: Vector) -> Vector:
    return v - n * (2.0 * v.dot(n))

---
### Let’s create a more complex scene

This scene shows several spheres with different levels of reflection and transparency. It is useful for observing how recursive reflection and refraction work.

The floor is slightly reflective, so you can also see the spheres reflected on the ground. The scene uses a point light and an ambient light. A skybox can be used as a simple background color source for rays that miss all objects.

In this project, the HDR skybox is not used as a physically correct HDR environment map. It is only sampled as an RGB background color, which can also appear in reflections and refractions.

> **Try it yourself:** Download a free HDR skybox and test it in the scene. The skybox files are not included in the repository because they can be large. You can find free HDR skyboxes on websites such as [HDRI Haven](https://hdrihaven.com/), often in very high resolutions.

In [ ]:
reflective_transparent_objects = [

    #red semi-reflective sphere
    Object(
        geometry=Sphere(),
        material=PhongMaterial(
            base_color=Color(1, 0, 0),
            reflectivity=0.2,
            transparency=0.0,
        ),
    ).translate(-3.0, 0.5, -6),

    # glass sphere with ior 1.5 and high transparency
    Object(
        geometry=Sphere(),
        material=PhongMaterial(
            base_color=Color(0.60, 0.60, 1.0),
            transparency=0.95,
            ior=1.5,
        ),
    ).translate(0.0, 0.5, -5.5),

    # mirror-like sphere
    Object(
        geometry=Sphere(),
        material=PhongMaterial(
            base_color=Color(1.0, 1.0, 1.0),
            reflectivity=0.9,
            transparency=0.0,
        ),
    ).translate(3.0, 0.5, -6.5),

    # blue sphere half reflective
    Object(
        geometry=Sphere(),
        material=PhongMaterial(base_color=Color(0, 0, 1), reflectivity=0.5),
    ).translate(-1, 0.25, -9.0),

    # green sphere with transparency set to 0.5 and high ior diamond like
    Object(
        geometry=Sphere(),
        material=PhongMaterial(base_color=Color(0, 1, 0), transparency=0.7, ior=2.418),
    ).translate(1.0, 0.25, -7.5),

    # floor
    Object(
        geometry=Plane(),
        material=PhongMaterial(
            base_color=Color(0.45, 0.45, 0.45),
            reflectivity=0.1,
        ),
    ).translate(0, -1, 0),
]

scene = Scene(
    objects=reflective_transparent_objects,
    lights=lights,
    camera=camera,
    skybox="./skyboxes/sand.hdr",
)

recursive_integrator = MyWhittedIntegrator(scene=scene)

## Comparison of recursion depths

### Depth 0 — local shading only

At depth 0, only the local shading model is used. Recursive reflection and refraction are not traced yet, so mirror and glass effects are not visible, only local shading at the first hit point. The result is therefore similar to the simple ray caster.

In [ ]:
rt = LinearRenderLoop(
    integrator=recursive_integrator,
    scene=scene,
    render_config=RenderConfig(
        resolution=Resolution.R360p,
        max_depth=0,
        samples_per_pixel=1,
    ),
    preview_config=PreviewConfig(
        progress_display=ProgressDisplay.TQDM_IMAGE_PREVIEW,
    )
)
img = rt.render("images/depth_0.png")
ipynb_display_images(img)

### Depth 1 — first recursive level

At depth 1, the first reflected and refracted rays are traced. Refraction becomes visible, but some reflections are still missing or incomplete because deeper recursive paths are not evaluated yet.

> **Note:** Some rays refracted into the glass sphere need more recursion steps before they can contribute visible reflections.

In [ ]:
rt = LinearRenderLoop(
    integrator=recursive_integrator,
    scene=scene,
    render_config=RenderConfig(
        resolution=Resolution.R360p,
        max_depth=1,
        samples_per_pixel=1,
    ),
    preview_config=PreviewConfig(
        progress_display=ProgressDisplay.TQDM_IMAGE_PREVIEW,
    )
)

img = rt.render("images/depth_1.png")
ipynb_display_images(img)

### Depth 2 — second recursive level

At depth 2 you can see more reflections and refractions.

In [ ]:
rt = LinearRenderLoop(
    integrator=recursive_integrator,
    scene=scene,
    render_config=RenderConfig(
        resolution=Resolution.R360p,
        max_depth=2,
        samples_per_pixel=1,
    ),
    preview_config=PreviewConfig(
        progress_display=ProgressDisplay.TQDM_IMAGE_PREVIEW,
    )
)

img = rt.render("images/depth_2.png")
ipynb_display_images(img)

### Depth 7 — multiple recursive levels

At depth 7, several levels of reflection and refraction are visible. The image looks more complete and visually more accurate, because more light paths are traced through the scene. This is especially noticeable in interactions between reflective and transparent objects.

The trade-off is higher computational cost, since each additional recursion level increases the number of rays that may need to be evaluated.

> **Try it yourself:** Change the `max_depth` value and observe how the image changes. You can also modify material properties such as reflectivity and transparency, or build a more complex scene with multiple reflective and transparent objects.

In [ ]:
rt = LinearRenderLoop(
    integrator=recursive_integrator,
    scene=scene,
    render_config=RenderConfig(
        resolution=Resolution.R360p,
        max_depth=7,
        samples_per_pixel=1,
    ),
    preview_config=PreviewConfig(
        progress_display=ProgressDisplay.TQDM_IMAGE_PREVIEW,
    )
)

img = rt.render("images/depth_7.png")
ipynb_display_images(img)

---
# Summary

In this notebook, we extended the ray tracer with recursive ray tracing and completed the main rendering system with a Whitted-style integrator. This integrator supports both reflection and refraction based on material properties.

We also implemented a simpler reflection-only integrator to demonstrate the basic idea of recursion in ray tracing before moving to the more complex case with transparent materials.

- **Ray caster** — a simple integrator that only computes local shading at the first hit point without recursion
- **Reflection integrator** — Simple recursive reflections based on material reflectivity
- **Whitted-style integrator** — recursive reflection and refraction based on material properties and Fresnel effect
- **Fresnel effect simplified with Schlick's approximation** — blending reflection and refraction based on angle of incidence
- **Recursion depth** — Examples of how increasing recursion depth allows more complex light paths to be captured, improving realism but increasing render time
- **Reflective and transparent scenes** — testing how glass, mirrors, and reflective floors interact with recursive ray tracing

**Next:**

- **Procedural textures** — generating surface detail without image files
- **Noise functions** — using Perlin noise, fBM, turbulence, and ridge noise
- **Material variation** — changing color, shininess, and surface appearance using noise
- **Normal perturbation** — modifying surface normals to create small visual bumps
- **Rock, marble, and wood materials** — examples of procedural materials built from noise

---

Below is a summary table comparing the three integrators implemented in this notebook:

| Integrator | Local shading | Reflection | Refraction | Fresnel term | Worst-case cost per primary ray |
|---|---:|---:|---:|---:|---|
| `MyRayCaster` | ✅ | ❌ | ❌ | ❌ | `O(1)` intersection queries |
| `MyReflectionIntegrator` | ✅ | ✅ | ❌ | ❌ | `O(d)`, linear in recursion depth |
| `MyWhittedIntegrator` | ✅ | ✅ | ✅ | ✅ | `O(2^d)`, exponential in recursion depth in the worst case |